In [1]:
import heapq
from itertools import count


class PuzzleState:
    def __init__(self, board, g_cost, goal_positions, parent=None, action=None):
        self.board = tuple(board)
        self.g_cost = g_cost
        self.parent = parent
        self.action = action
        self.goal_positions = goal_positions
        self.h_cost = self.manhattan_distance()
        self.f_cost = self.g_cost + self.h_cost

    def manhattan_distance(self):
        distance = 0
        for index, value in enumerate(self.board):
            if value != 0:
                current_row, current_col = divmod(index, 3)
                goal_row, goal_col = self.goal_positions[value]
                distance += abs(current_row - goal_row) + abs(current_col - goal_col)
        return distance

    def get_neighbors(self):
        neighbors = []
        blank_index = self.board.index(0)
        row, col = divmod(blank_index, 3)

        # These describe how the blank space moves.
        moves = [
            (-1, 0, "Up"),
            (1, 0, "Down"),
            (0, -1, "Left"),
            (0, 1, "Right"),
        ]

        for dr, dc, move_name in moves:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank_index = new_row * 3 + new_col
                new_board = list(self.board)
                new_board[blank_index], new_board[new_blank_index] = (
                    new_board[new_blank_index],
                    new_board[blank_index],
                )
                neighbors.append(
                    PuzzleState(
                        new_board,
                        self.g_cost + 1,
                        self.goal_positions,
                        parent=self,
                        action=move_name,
                    )
                )
        return neighbors


def print_board(board):
    for row in range(3):
        values = board[row * 3:(row + 1) * 3]
        print(" ".join("_" if value == 0 else str(value) for value in values))


def count_inversions(board):
    values = [value for value in board if value != 0]
    return sum(
        values[i] > values[j]
        for i in range(len(values))
        for j in range(i + 1, len(values))
    )


def is_solvable(start, goal):
    # For a 3x3 puzzle, start and goal are mutually reachable
    # when their inversion counts have the same parity.
    return count_inversions(start) % 2 == count_inversions(goal) % 2


def solve_8_puzzle(initial_board, goal_board):
    goal_positions = {
        value: divmod(index, 3)
        for index, value in enumerate(goal_board)
    }

    initial_state = PuzzleState(initial_board, 0, goal_positions)

    # Add a sequence number to break ties safely when f-costs match.
    sequence = count()
    open_heap = [(initial_state.f_cost, next(sequence), initial_state)]
    best_g = {initial_state.board: 0}
    expanded = set()

    while open_heap:
        _, _, current = heapq.heappop(open_heap)

        # Ignore an outdated entry if a better route has already been found.
        if current.g_cost != best_g.get(current.board):
            continue

        if current.board == tuple(goal_board):
            path = []
            state = current
            while state is not None:
                path.append(state)
                state = state.parent
            return list(reversed(path))

        if current.board in expanded:
            continue
        expanded.add(current.board)

        for neighbor in current.get_neighbors():
            if neighbor.g_cost < best_g.get(neighbor.board, float("inf")):
                best_g[neighbor.board] = neighbor.g_cost
                heapq.heappush(
                    open_heap,
                    (neighbor.f_cost, next(sequence), neighbor),
                )

    return None


def get_board_input(prompt):
    print(prompt)
    print("Enter 9 numbers (0-8), separated by spaces. Use 0 for the blank.")
    while True:
        try:
            values = list(map(int, input("Input: ").split()))
            if len(values) != 9:
                print("Please enter exactly 9 numbers.")
                continue
            if sorted(values) != list(range(9)):
                print("Use each number from 0 to 8 exactly once.")
                continue
            return tuple(values)
        except ValueError:
            print("Invalid input. Please enter numbers only.")


def main():
    print("=" * 45)
    print("         8-PUZZLE SOLVER USING A*")
    print("=" * 45)

    initial = get_board_input("\n--- ENTER INITIAL STATE ---")
    goal = get_board_input("\n--- ENTER GOAL STATE ---")

    print("\nInitial state:")
    print_board(initial)
    print("\nGoal state:")
    print_board(goal)

    if not is_solvable(initial, goal):
        print("\nError: This initial state cannot reach the given goal state.")
        return

    print("\nSolving using A* Search with Manhattan Distance...")
    path = solve_8_puzzle(initial, goal)

    if path is None:
        print("No path to the goal was found.")
        return

    print(f"\nSuccess! Optimal solution requires {len(path) - 1} moves.")
    print("\n========== STEP-BY-STEP PATH ==========")

    for step, state in enumerate(path):
        if step == 0:
            print("\nSTEP 0: INITIAL STATE")
        else:
            print(f"\nSTEP {step}: Move blank space '{state.action.upper()}'")

        print_board(state.board)
        print(
            f"g(n) = {state.g_cost} | "
            f"h(n) = {state.h_cost} | "
            f"f(n) = {state.f_cost}"
        )
        print("-" * 35)

    print("\nGOAL STATE SUCCESSFULLY REACHED!")


if __name__ == "__main__":
    main()


         8-PUZZLE SOLVER USING A*

--- ENTER INITIAL STATE ---
Enter 9 numbers (0-8), separated by spaces. Use 0 for the blank.
Input: 2 3 1 4 5 0 8 6 7

--- ENTER GOAL STATE ---
Enter 9 numbers (0-8), separated by spaces. Use 0 for the blank.
Input: 1 2 3 4 0 5 6 7 8

Initial state:
2 3 1
4 5 _
8 6 7

Goal state:
1 2 3
4 _ 5
6 7 8

Solving using A* Search with Manhattan Distance...

Success! Optimal solution requires 21 moves.

========== STEP-BY-STEP PATH ==========

STEP 0: INITIAL STATE
2 3 1
4 5 _
8 6 7
g(n) = 0 | h(n) = 9 | f(n) = 9
-----------------------------------

STEP 1: Move blank space 'UP'
2 3 _
4 5 1
8 6 7
g(n) = 1 | h(n) = 10 | f(n) = 11
-----------------------------------

STEP 2: Move blank space 'LEFT'
2 _ 3
4 5 1
8 6 7
g(n) = 2 | h(n) = 9 | f(n) = 11
-----------------------------------

STEP 3: Move blank space 'LEFT'
_ 2 3
4 5 1
8 6 7
g(n) = 3 | h(n) = 8 | f(n) = 11
-----------------------------------

STEP 4: Move blank space 'DOWN'
4 2 3
_ 5 1
8 6 7
g(n) = 4 | h